In [ ]:
# EuroSAT Land Use Classification with EfficientNet-B3
# Simple, focused implementation with Gradio UI

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import requests
from io import BytesIO
import gradio as gr

# ============================================================================
# SECTION 1: Setup and Configuration
# ============================================================================

# Mount Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully!")
    DRIVE_MOUNTED = True
except ImportError:
    print("ℹ️ Not running in Colab - using local file system")
    DRIVE_MOUNTED = False

# EuroSAT class names
EUROSAT_CLASSES = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial',
    'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake'
]

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Using device: {device}")

# ============================================================================
# SECTION 2: Model Architecture
# ============================================================================

class EuroSATEfficientNetB3(nn.Module):
    """
    EfficientNet-B3 model customized for EuroSAT classification

    Architecture:
    - Backbone: EfficientNet-B3 pretrained on ImageNet
    - Modified classifier head for 10 classes
    - Total parameters: ~12M
    - Best validation accuracy: 98.6%

    Key Features:
    - Compound scaling (depth, width, resolution)
    - Mobile Inverted Bottleneck Convolution (MBConv)
    - Squeeze-and-Excitation blocks
    - Efficient parameter usage
    """
    def __init__(self, num_classes=10):
        super(EuroSATEfficientNetB3, self).__init__()

        # Load pretrained EfficientNet-B3
        self.model = models.efficientnet_b3(
            weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1
        )

        # Replace classifier head for EuroSAT (10 classes)
        in_features = self.model.classifier[1].in_features  # 1536
        self.model.classifier[1] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.model(x)

# ============================================================================
# SECTION 3: Image Preprocessing
# ============================================================================

def get_transforms():
    """
    Get image preprocessing transforms for EfficientNet-B3

    Transforms:
    1. Resize to 224x224 (EfficientNet-B3 input size)
    2. Convert to tensor
    3. Normalize with ImageNet mean and std

    Note: EfficientNet-B3 native resolution is 300x300, but we use 224x224
    to match the training configuration from the EuroSAT dataset
    """
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],  # ImageNet mean
            std=[0.229, 0.224, 0.225]     # ImageNet std
        )
    ])

# ============================================================================
# SECTION 4: Model Loading
# ============================================================================

def load_model(model_path):
    """
    Load EfficientNet-B3 model from checkpoint

    Args:
        model_path: Path to .pth model file

    Returns:
        model: Loaded model in eval mode
        info: Dictionary with model information

    Note: Handles different checkpoint formats:
    - Standard format with 'model_state_dict'
    - Direct state dict with 'features.' and 'classifier.' keys
    - Format with 'classes' key
    """
    try:
        # Handle Google Drive paths
        if DRIVE_MOUNTED and not model_path.startswith('/content/drive/'):
            if not model_path.startswith('/'):
                model_path = '/content/drive/MyDrive/' + model_path

        # Load checkpoint
        print(f"📂 Loading model from: {model_path}")
        checkpoint = torch.load(model_path, map_location=device)

        # Create model
        model = EuroSATEfficientNetB3(num_classes=len(EUROSAT_CLASSES))

        # Handle different checkpoint formats
        if 'model_state_dict' in checkpoint:
            # Standard format with model_state_dict wrapper
            state_dict = checkpoint['model_state_dict']
            accuracy = checkpoint.get('val_accuracy', checkpoint.get('test_acc', '98.6'))
            epoch = checkpoint.get('epoch', 'Unknown')
        elif 'classes' in checkpoint:
            # Format from training code with classes key
            state_dict = {k: v for k, v in checkpoint.items() if k != 'classes'}
            accuracy = '98.6'  # From training document
            epoch = '12'
        else:
            # Direct state dict
            state_dict = checkpoint
            accuracy = '98.6'  # From training document
            epoch = '12'

        # Fix key mismatch: add "model." prefix if needed
        corrected_state_dict = {}
        for key, value in state_dict.items():
            if key.startswith('features.') or key.startswith('classifier.'):
                # Add "model." prefix to match the model structure
                new_key = f"model.{key}"
                corrected_state_dict[new_key] = value
            else:
                # Keep other keys as they are
                corrected_state_dict[key] = value

        # Load the corrected state dict
        model.load_state_dict(corrected_state_dict)

        # Move to device and set to eval mode
        model.to(device)
        model.eval()

        # Model info
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

        info = {
            'accuracy': accuracy,
            'epoch': epoch,
            'total_params': total_params,
            'trainable_params': trainable_params
        }

        print(f"✅ Model loaded successfully!")
        print(f"   Accuracy: {accuracy}%")
        print(f"   Epoch: {epoch}")
        print(f"   Parameters: {total_params:,}")

        return model, info

    except Exception as e:
        print(f"❌ Error loading model: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, {'error': str(e)}

# ============================================================================
# SECTION 5: Prediction Function
# ============================================================================

def predict(model, image, transform):
    """
    Make prediction on an image using EfficientNet-B3

    Args:
        model: Loaded PyTorch model
        image: PIL Image
        transform: Image preprocessing transforms

    Returns:
        results: Dictionary with prediction results including:
            - predicted_class: Top predicted class name
            - confidence: Confidence score (0-1)
            - all_probabilities: Array of all class probabilities
            - top5_classes: List of top 5 class names
            - top5_probs: List of top 5 probabilities
    """
    try:
        # Preprocess image
        image_tensor = transform(image).unsqueeze(0).to(device)

        # Make prediction
        with torch.no_grad():
            outputs = model(image_tensor)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)

            # Get results
            probs = probabilities[0].cpu().numpy()
            predicted_idx = torch.argmax(outputs, dim=1).item()
            confidence = probabilities[0][predicted_idx].item()
            predicted_class = EUROSAT_CLASSES[predicted_idx]

            # Get top 5 predictions
            top5_indices = np.argsort(probs)[-5:][::-1]
            top5_classes = [EUROSAT_CLASSES[i] for i in top5_indices]
            top5_probs = [probs[i] for i in top5_indices]

            return {
                'predicted_class': predicted_class,
                'confidence': confidence,
                'all_probabilities': probs,
                'top5_classes': top5_classes,
                'top5_probs': top5_probs
            }

    except Exception as e:
        print(f"❌ Prediction error: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# ============================================================================
# SECTION 6: Visualization
# ============================================================================

def create_bar_chart(results):
    """
    Create bar chart of top 5 predictions with confidence percentages
    """
    if results is None:
        return None

    fig, ax = plt.subplots(figsize=(10, 6))

    classes = results['top5_classes']
    probs = [p * 100 for p in results['top5_probs']]

    # Color the top prediction green, others blue
    colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(classes))]
    bars = ax.barh(classes, probs, color=colors)

    # Add percentage labels
    for i, (bar, prob) in enumerate(zip(bars, probs)):
        ax.text(prob + 1, bar.get_y() + bar.get_height()/2,
                f'{prob:.1f}%', va='center', fontweight='bold')

    ax.set_xlabel('Confidence (%)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Land Use Class', fontsize=12, fontweight='bold')
    ax.set_title('Top 5 Predictions - EfficientNet-B3', fontsize=14, fontweight='bold')
    ax.set_xlim(0, 105)
    ax.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    return fig

def create_probability_heatmap(results):
    """
    Create heatmap showing all class probabilities
    """
    if results is None:
        return None

    fig, ax = plt.subplots(figsize=(12, 4))

    probs = results['all_probabilities'] * 100
    probs = probs.reshape(1, -1)

    im = ax.imshow(probs, cmap='YlGnBu', aspect='auto', vmin=0, vmax=100)

    ax.set_xticks(range(len(EUROSAT_CLASSES)))
    ax.set_xticklabels(EUROSAT_CLASSES, rotation=45, ha='right')
    ax.set_yticks([])
    ax.set_title('All Class Probabilities (%)', fontsize=14, fontweight='bold')

    # Add text annotations
    for i, prob in enumerate(probs[0]):
        text_color = 'white' if prob > 50 else 'black'
        ax.text(i, 0, f'{prob:.1f}%', ha='center', va='center',
                color=text_color, fontweight='bold')

    plt.colorbar(im, ax=ax, label='Probability (%)')
    plt.tight_layout()
    return fig

# ============================================================================
# SECTION 7: Gradio Interface
# ============================================================================

# Global model variable
loaded_model = None
transform = get_transforms()

def load_model_gradio(model_path):
    """Load model for Gradio interface"""
    global loaded_model

    if not model_path or not model_path.strip():
        return "⚠️ Please provide a model path"

    loaded_model, info = load_model(model_path.strip())

    if loaded_model is None:
        return f"❌ Failed to load model: {info.get('error', 'Unknown error')}"

    return f"""✅ Model loaded successfully!

📊 Model Information:
• Architecture: EfficientNet-B3
• Accuracy: {info['accuracy']}%
• Epoch: {info['epoch']}
• Total Parameters: {info['total_params']:,}
• Trainable Parameters: {info['trainable_params']:,}

🏆 EfficientNet-B3 Advantages:
• Compound scaling for balanced efficiency
• Squeeze-and-Excitation blocks for channel attention
• Mobile Inverted Bottleneck Convolutions (MBConv)
• State-of-the-art accuracy with fewer parameters

Ready to make predictions!"""

def predict_gradio(image):
    """Make prediction for Gradio interface"""
    global loaded_model

    if loaded_model is None:
        return None, "❌ Please load a model first", None, None

    if image is None:
        return None, "⚠️ Please provide an image", None, None

    # Make prediction
    results = predict(loaded_model, image, transform)

    if results is None:
        return None, "❌ Prediction failed", None, None

    # Create results text
    results_text = f"""✅ Prediction Complete!

🎯 Predicted Class: {results['predicted_class']}
📈 Confidence: {results['confidence']*100:.2f}%

📊 Top 5 Predictions:
"""
    for i, (cls, prob) in enumerate(zip(results['top5_classes'], results['top5_probs']), 1):
        emoji = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "📍"
        results_text += f"{emoji} {cls}: {prob*100:.1f}%\n"

    # Create charts
    bar_chart = create_bar_chart(results)
    heatmap = create_probability_heatmap(results)

    return image, results_text, bar_chart, heatmap

def load_from_url(url):
    """Load image from URL"""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert('RGB')
        return image
    except Exception as e:
        print(f"Error loading image from URL: {e}")
        return None

# Create Gradio interface
def create_interface():
    with gr.Blocks(theme=gr.themes.Soft(), title="EuroSAT EfficientNet-B3 Classifier") as demo:

        gr.Markdown("""
        # 🛰️ EuroSAT Land Use Classification with EfficientNet-B3

        Upload a satellite or aerial image to classify it into one of 10 land use categories.

        **EfficientNet-B3** achieves **98.6% validation accuracy** on the EuroSAT dataset!

        **Classes**: AnnualCrop, Forest, HerbaceousVegetation, Highway, Industrial,
        Pasture, PermanentCrop, Residential, River, SeaLake
        """)

        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 🔧 Step 1: Load Model")

                model_path_input = gr.Textbox(
                    label="Model Path",
                    placeholder="/content/drive/MyDrive/saved_models/efficientnet_b3_eurosat_best.pth",
                    value="/content/drive/MyDrive/saved_models/efficientnet_b3_eurosat_best.pth",
                    info="Path to your EfficientNet-B3 model weights (.pth file)"
                )

                load_btn = gr.Button("📥 Load Model", variant="primary")
                model_status = gr.Textbox(
                    label="Model Status",
                    lines=12,
                    interactive=False
                )

                gr.Markdown("""
                ### 📝 Path Examples:
                - Google Drive: `/content/drive/MyDrive/saved_models/model.pth`
                - Weights folder: `/content/drive/MyDrive/weights/model.pth`
                - Relative: `saved_models/model.pth`

                ### 🏆 Model Highlights:
                - **Accuracy**: 98.6% (best among tested models)
                - **Parameters**: ~12M (efficient design)
                - **Training**: 12 epochs with early stopping
                - **Architecture**: Compound-scaled EfficientNet
                """)

            with gr.Column(scale=1):
                gr.Markdown("### 🖼️ Step 2: Upload Image")

                image_input = gr.Image(
                    label="Upload Image",
                    type="pil",
                    height=300
                )

                gr.Markdown("**Or load from URL:**")
                image_url = gr.Textbox(
                    label="Image URL",
                    placeholder="https://example.com/satellite-image.jpg"
                )
                load_url_btn = gr.Button("🔗 Load from URL")

                predict_btn = gr.Button("🔍 Classify Image", variant="primary", size="lg")

        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 📊 Results")
                results_text = gr.Textbox(
                    label="Prediction Results",
                    lines=13,
                    interactive=False
                )

            with gr.Column(scale=1):
                gr.Markdown("### 📈 Top 5 Predictions")
                bar_chart = gr.Plot(label="Confidence Bar Chart")

        with gr.Row():
            gr.Markdown("### 🎨 All Class Probabilities")

        with gr.Row():
            heatmap = gr.Plot(label="Probability Heatmap", show_label=False)

        gr.Markdown("""
        ---
        ### 📖 Instructions:
        1. **Load Model**: Enter the path to your trained EfficientNet-B3 model and click "Load Model"
        2. **Upload Image**: Either upload an image file or paste a URL
        3. **Classify**: Click "Classify Image" to see the prediction

        ### 🔬 EfficientNet-B3 Architecture:
        - **Compound Scaling**: Balances depth, width, and resolution
        - **MBConv Blocks**: Mobile Inverted Bottleneck Convolutions
        - **Squeeze-and-Excitation**: Channel-wise attention mechanism
        - **Input Size**: 224×224 pixels (auto-resized from images)
        - **Output**: Softmax probabilities for 10 land use classes

        ### 📊 Performance:
        - Validation Accuracy: **98.6%**
        - Training Epochs: 12 (with early stopping)
        - Parameters: ~12M (highly efficient)
        - Best performer among ResNet50, DenseNet-121, and Vision Transformer

        ### 🎯 Use Cases:
        - Land use mapping and monitoring
        - Urban planning and development
        - Agricultural analysis
        - Environmental monitoring
        - Change detection in satellite imagery
        """)

        # Event handlers
        load_btn.click(
            fn=load_model_gradio,
            inputs=[model_path_input],
            outputs=[model_status]
        )

        load_url_btn.click(
            fn=load_from_url,
            inputs=[image_url],
            outputs=[image_input]
        )

        predict_btn.click(
            fn=predict_gradio,
            inputs=[image_input],
            outputs=[image_input, results_text, bar_chart, heatmap]
        )

    return demo

# ============================================================================
# SECTION 8: Launch
# ============================================================================

if __name__ == "__main__":
    print("=" * 70)
    print("🛰️ EuroSAT LAND USE CLASSIFICATION - EfficientNet-B3")
    print("=" * 70)
    print(f"Device: {device}")
    print(f"Google Drive: {'✅ Mounted' if DRIVE_MOUNTED else '❌ Not mounted'}")
    print(f"Classes: {len(EUROSAT_CLASSES)}")
    print(f"Model: EfficientNet-B3 (98.6% accuracy)")
    print("=" * 70)

    # Create and launch interface
    demo = create_interface()
    demo.launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=True,
        debug=True
    )

Mounted at /content/drive
✅ Google Drive mounted successfully!
🔧 Using device: cuda
🛰️ EuroSAT LAND USE CLASSIFICATION - EfficientNet-B3
Device: cuda
Google Drive: ✅ Mounted
Classes: 10
Model: EfficientNet-B3 (98.6% accuracy)
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1b7f8c48815f43f997.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


📂 Loading model from: /content/drive/MyDrive/weights/efficientnet_b3_eurosat_best.pth
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 201MB/s]


✅ Model loaded successfully!
   Accuracy: 98.6%
   Epoch: Unknown
   Parameters: 10,711,602
